# Main Quest 03 · 공간 한글 자모 생성 연구

- 목적: 일반 한글을 하나의 고정된 자모 배치로 출력하는 소형 디코더 모델 비교
- 최종 논문: [영문 IEEEtran PDF](온19기_MainQuest3_강지수.pdf)
- 이 노트북: 실제 저장된 결과를 읽고 다시 채점 · GPU 학습 자동 시작 없음
- 결과: C 위치 F1 28.41%, B 22.16% · 두 조건 완전 정답 0/25
- 최초 5회 실험: [별도 노트북](Original_5epoch.ipynb) · 선택된 모델은 1회차

## 1. 이론과 연구 질문

- 한국어 추론 능력과 자모 배치 생성 능력을 구분
- 자모 입력 오타 연구에서 출력 문자 보존 문제로 평가 관점 확장
- 시각 텍스트 읽기와 배치 쓰기는 서로 다른 과업
- 좌표 정보가 유용할 수 있다는 선행 연구를 근거로 좌표 보조학습 검토
- 주 질문: C(변환+좌표)가 B(변환+목록)보다 완전한 배치 생성을 개선하는가?
- 결론: 부분 위치 일치는 높지만 완전 정답 개선은 미확인

## 2. 실행 위치

저장소를 내려받아 `MainQuest/Quest03` 폴더에서 실행합니다. Colab에서는 아래 예시로 저장소를 복제한 뒤 해당 폴더로 이동합니다. 현재 블록은 설명용이며 자동 실행하지 않습니다.

```python
!git clone https://github.com/kritik-sowieso/AIFFEL_quest_rs.git
%cd AIFFEL_quest_rs/MainQuest/Quest03
```


In [1]:
# 현재 폴더에 실제 결과 파일이 있는지 먼저 확인합니다.
from pathlib import Path
import json, subprocess, sys
root=Path.cwd()
assert (root/'extension_v2/report/results.json').exists(), 'MainQuest/Quest03 폴더에서 실행하세요.'
report=json.loads((root/'extension_v2/report/results.json').read_text())
print('학습 198 / 검증 25 / 테스트 25 · 고정 학습 진단 12')
print('D/B/C 각각 10 epoch(자료 전체를 보는 회차) · 고정 5/10회 비교')

학습 198 / 검증 25 / 테스트 25 · 고정 학습 진단 12
D/B/C 각각 10 epoch(자료 전체를 보는 회차) · 고정 5/10회 비교


## 3. 원시 출력 검수

- 원문·정답·분할·모델 버전·파일 지문 대조
- 원시 출력 그대로 재채점 · 정답으로 교정하지 않음
- 학습 표본 12개는 일반화 평가가 아닌 학습 적합 진단


In [2]:
# 저장 출력만 읽으므로 GPU가 필요하지 않습니다.
checked=subprocess.run([sys.executable,'-m','evaluation.verify_public_extension'],check=True,capture_output=True,text=True)
print(checked.stdout)

검수 통과: 추가 실험 18개 파일 · 372개 출력 · 기반 모델 A · 저장 집계와 일치
기존 테스트 재사용 · 탐색 결과 · 추가 학습 없음



## 4. 구성요소 비교(ablation study)

- A: 추가학습 전 기반 모델
- D: 변환만 학습
- B: 변환 + 자모 목록 보조학습
- C: 변환 + 자모·좌표 보조학습
- 완전 정답: 전체 출력 일치 / 문자 F1: 문자 구성 일치 / 위치 F1: 문자·좌표 일치


In [3]:
# 10회차를 주 비교로 고정했습니다. 점수를 보고 회차를 바꾸지 않습니다.
for condition in ['A','D','B','C']:
    info=report['baseline_A_reused'] if condition=='A' else report['conditions'][condition]['10']
    v=info['test']['overall']
    print(condition, '정답', str(v['exact_grid'])+'/25',
          '문자 F1', round(v['inventory_f1']*100,2),
          '위치 F1', round(v['coordinate_f1']*100,2))

A 정답 0/25 문자 F1 0.16 위치 F1 0.16
D 정답 1/25 문자 F1 53.3 위치 F1 26.36
B 정답 0/25 문자 F1 55.98 위치 F1 22.16
C 정답 0/25 문자 F1 53.91 위치 F1 28.41


## 5. 학습량 확대와 학습 진단

최초 5회 실험의 검증 선택 모델(1회차)과 이번 고정 5회차 결과는 다릅니다.


In [4]:
for condition in ['D','B','C']:
    for epoch in ['5','10']:
        t=report['conditions'][condition][epoch]['test']['overall']
        print(condition,epoch,'회차:', '문자 F1',round(t['inventory_f1']*100,2),
              '위치 F1',round(t['coordinate_f1']*100,2), '정답',str(t['exact_grid'])+'/25')
    v=report['conditions'][condition]['10']['train_probe']['overall']
    print('고정 학습 표본 진단:',str(v['exact_grid'])+'/12')

D 5 회차: 문자 F1 34.89 위치 F1 16.23 정답 0/25
D 10 회차: 문자 F1 53.3 위치 F1 26.36 정답 1/25
고정 학습 표본 진단: 1/12
B 5 회차: 문자 F1 40.69 위치 F1 16.87 정답 0/25
B 10 회차: 문자 F1 55.98 위치 F1 22.16 정답 0/25
고정 학습 표본 진단: 0/12
C 5 회차: 문자 F1 52.57 위치 F1 20.0 정답 0/25
C 10 회차: 문자 F1 53.91 위치 F1 28.41 정답 0/25
고정 학습 표본 진단: 1/12


## 6. 오류 원문 보기

공백과 줄바꿈을 보존하기 위해 `repr` 표기로 확인합니다.


In [5]:
for condition in ['D','B','C']:
    item=next(x for x in report['conditions'][condition]['10']['test']['items'] if x['text']=='기린')
    print(condition,'입력',item['text'],'정답',repr(item['target']),'출력',repr(item['prediction']))

D 입력 기린 정답 'ㄱㅣ ㄹㅣ\n   ㄴ' 출력 'ㄱ ㄹㅣ\nㅜ ㄴ'
B 입력 기린 정답 'ㄱㅣ ㄹㅣ\n   ㄴ' 출력 'ㄱ ㄹㅣ\nㅜ ㄴ'
C 입력 기린 정답 'ㄱㅣ ㄹㅣ\n   ㄴ' 출력 'ㄱ ㄹㅣ\nㅜ ㄴ'


## 7. 학습·평가 흐름

```text
일반 한글 → 규칙으로 합성 정답·좌표 생성 → 198/25/25 고정 분할
                     ↓
       동일 Qwen2.5-0.5B 기반 D / B / C
                     ↓
         각 10회 학습 → 고정 5/10회 평가
                     ↓
         원시 출력 372개 → 집계·오류 분석
                     ↓
       영문 논문 → Overleaf → GitHub 제출
```

![시스템](paper/figures/architecture_en.png)
![학습량과 위치 점수](paper/figures/training_and_layout.png)

## 8. 재학습 안내 · 이 노트북에서 실행하지 않음

GPU에서 **새 작업 폴더**에 training/evaluation/jamo/tests/data/extension_v2 코드를 준비하고 기존 `checkpoints` 및 `runs/extension`이 없는지 확인합니다. 파일 지문이 고정 계획과 맞아야 합니다.

```python
!pip install -r requirements-training.txt
!python -m training.reproduce_extension --run-training
```

- 별도 재현 실행기가 10회 학습·5/10회 평가 수행 · 90분 상한
- 기존 결과를 덮어쓰지 않음 · 이번 제출 준비에서는 재학습하지 않음
- 모델 어댑터·옵티마이저 갱신과 생성 평가의 개념: [설명](extension_v2/읽는_순서.md)

## 9. 해석과 회고

- 배운 점: 손실 하락, 부분 F1 증가, 완전한 문구 변환 성공은 서로 다름
- OCR이 독립 자모를 안정적으로 읽지 못해 합성 좌표 감독으로 검증 범위 축소
- 학습 표본에서도 대부분 실패하여 일반화만으로 문제를 설명하기 어려움
- 보조 과제의 종류에 따라 문자 보존과 위치 배치 결과가 다름
- 한계: 단일 시드, 작은 표본, 기존 테스트 재사용, 계산량 차이
- 후속: 새 테스트 세트, 반복 시드, 계산량 통제, 구조를 제한한 출력 방식
- 데모: 기존 B 1회차 유지 · 최신 실험 모델로 교체하지 않음
